Language: **English (default)** | [Polski](lab.ipynb)

# Lab 11. An inference pipeline for a small language model

This notebook accompanies [`README.md`](README.md) and covers the complete
pipeline from prompt and tokenization to generation, decoding, measurement,
and evaluation.

**Sections 0-11 form the mandatory 105-minute core. The remaining 15 minutes
serve as a buffer.** Sections R1-R2 at the end of the notebook are optional.

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"

## 1. Model and tokenizer

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"load time: {time.time() - t0:.2f}s")
print(f"number of parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TODO: print model.config.num_attention_heads and model.config.num_key_value_heads
# Are they equal? What does this mean for KV cache memory cost? (see the lecture)
raise NotImplementedError("TODO: check Q vs KV heads")

## 2. Preparing prompts

Prepare **exactly 3 required** prompts:
1. **one** unambiguous "information extraction" prompt (has a precise
   expected answer),
2. **a second** unambiguous prompt, of a different type (e.g. "format
   transformation"),
3. **one qualitative** prompt (a paraphrase or continuation,
   `expected=None`).

For **every** prompt with an unambiguous answer, record that answer
**before running the model**. The required structure is a
list of dicts `{"id", "type", "prompt", "expected"}`.

**Extension R1 (optional):** add 2 more prompts (e.g. a question with
context, a continuation) with full treatment as in the core.

In [ ]:
# TODO: build a list of EXACTLY 3 required dicts {"id","type","prompt","expected"}:
# 1. one unambiguous prompt (e.g. information extraction), expected != None
# 2. a SECOND unambiguous prompt, of a different type (e.g. format transformation)
# 3. one QUALITATIVE prompt (a paraphrase or continuation), expected=None
prompts = []
raise NotImplementedError("TODO: preparing prompts")

## 3. Chat template

The `format_prompt(prompt_text)` function should return `inputs` ready for
`model.generate(**inputs, ...)`, using `tokenizer.apply_chat_template()`
with `add_generation_prompt=True`. **Do not** manually add any special
tokens. `apply_chat_template` already inserts them (see the lecture,
"Chat templates" section).

**API note.** `apply_chat_template(...,
tokenize=True, return_tensors="pt")` **without** `return_dict=True` may
return different types across `transformers` versions. Set
`return_dict=True` explicitly and access `inputs["input_ids"]` instead of
assuming a bare tensor.

Finally, verify that the function adds the special tokens. The decoded prompt
must start with the BOS token/
`<|im_start|>`, not with the prompt's content).

In [ ]:
def format_prompt(prompt_text):
    # TODO:
    # 1. build messages = [{"role": "user", "content": prompt_text}]
    # 2. return tokenizer.apply_chat_template(messages, tokenize=True,
    #    add_generation_prompt=True, return_tensors="pt", return_dict=True)
    raise NotImplementedError("TODO: format_prompt")

**Verification (a provided assertion): the decoded start of the
formatted prompt must begin with a special token (BOS/`<|im_start|>`),
not with the content.**

In [ ]:
check_inputs = format_prompt("test")  # noqa: F821 - defined in section 3
start_decoded = tokenizer.decode(check_inputs["input_ids"][0][:1])
print(f"first token of the formatted prompt: {start_decoded!r}")
assert start_decoded == tokenizer.bos_token, (
    f"BUG: expected {tokenizer.bos_token!r} as the first token, got {start_decoded!r}"
)
print("OK: format_prompt adds a special token at the start.")

## 4. Generation and correct decoding

The `generate_response(prompt_text, **gen_kwargs)` function should:
1. format the prompt (section 3),
2. call `model.generate(**inputs, **gen_kwargs, pad_token_id=...)`,
3. **extract ONLY the newly generated tokens** (not the whole `output`,
   which also contains the prompt; see the lecture, Experiment B/E),
4. decode (`skip_special_tokens=True`),
5. return the text **and** the number of newly generated tokens (needed
   for the throughput measurement in section 6).

In [ ]:
def generate_response(prompt_text, **gen_kwargs):
    # TODO:
    # 1. inputs = format_prompt(prompt_text)
    # 2. prompt_len = inputs["input_ids"].shape[1]
    # 3. out = model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    # 4. EXTRACT ONLY the newly generated tokens: out[0, prompt_len:] (NOT all of out!)
    # 5. decode (skip_special_tokens=True)
    # 6. return (text, number_of_new_tokens)
    raise NotImplementedError("TODO: generate_response")

**Verification (a provided assertion): for `do_sample=False`, two calls
with the same prompt must give an identical result.**

In [ ]:
r1, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
r2, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
print(f"run 1: {r1!r}")
print(f"run 2: {r2!r}")
assert r1 == r2, "BUG: greedy decoding should be deterministic"
print("OK: deterministic repeats are identical.")

## 5. Deterministic generation and sampling

For **all 3 required** prompts (section 2), generate a **greedy**
(`do_sample=False`) response. Additionally, for a **subset** (the
unambiguous prompt `extract` **and** the qualitative prompt `paraphrase`
but **not** `format`, to avoid requiring the full cross-product of 3
prompts x 3 strategies), also run **sampling** with two temperatures
(0.7 and 1.3). Use a **fixed seed** (`torch.manual_seed(...)`) before
each sampling call.

**Extension R2 (optional):** also run sampling for `format` (and for the
R1 prompts, if present). This shows whether sampling helps on unambiguous
tasks excluded from the core comparison.

In [ ]:
configs_greedy = [("greedy", dict(do_sample=False))]
configs_sampling = [
    ("sampling tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling tau=1.3", dict(do_sample=True, temperature=1.3)),
]
SAMPLING_SUBSET = {"extract", "paraphrase"}  # prompt ids that additionally get sampling

# TODO:
# for each prompt (section 2): always greedy; ADDITIONALLY sampling (both variants
# from configs_sampling) ONLY if p["id"] in SAMPLING_SUBSET
# 1. set torch.manual_seed(0) BEFORE every call (reproducibility)
# 2. measure the time (time.time() before/after) of the call
#    generate_response(prompt, max_new_tokens=40, **kwargs)
# 3. collect the results (a list of dicts: id, strategy, text, n_new, time, tok_s)
results = []
raise NotImplementedError("TODO: generation - greedy for all, sampling for the subset")

## 6. Measuring time and throughput

The timings were collected in section 5. Use them to print a tokens-per-second
table for every prompt and strategy combination in `results`.

In [ ]:
# TODO: print id, strategy, and tok/s for every entry in `results`
raise NotImplementedError("TODO: throughput table")

## 7. Effect of prompt length

Select **one** prompt and prepare a longer version, for example by adding
context or conversation history. Measure the generation time for **the same number**
of new tokens for the short and long versions of the prompt.

Set `min_new_tokens == max_new_tokens` to force **the same** number of new
tokens for both prompt lengths. Otherwise,
differences in generation time, such as early stopping at EOS, would
distort the measurement of prompt length alone.

In [ ]:
short_prompt = "What is 2+2?"
long_prompt = ("Here is a lot of context to pad out the prompt. " * 30) + " What is 2+2?"

# TODO: for both prompts, measure the time of generate_response with max_new_tokens=20 AND
# min_new_tokens=20 (the same number of new tokens for both!), print prompt_len and time
raise NotImplementedError("TODO: effect of prompt length")

## 8. KV cache

In [ ]:
# TODO: for use_cache=True and use_cache=False, measure the time of generate_response
# ("Tell me about the solar system.", max_new_tokens=40, do_sample=False, use_cache=...)
raise NotImplementedError("TODO: KV cache")

## 9. Evaluating quality

1. **Automatically**, for **both** prompts with an unambiguous answer
   (section 2), check their **greedy** responses with
   `p["expected"].lower() in response.lower()`.
2. **Qualitatively**, evaluate **every** strategy run for the qualitative
   prompt (greedy + 2 sampling variants from
   section 5) against the five criteria from the README (correct
   execution, completeness, format compliance, naturalness,
   reproducibility). Each rating needs a **specific justification based on a
   fragment of the answer**, not "looks good."

❌ "This answer looks better, so it is better."

✅ Separate criteria prevent the evaluation from resting on a single,
subjective impression.

**Extension R1/R2 (optional):** the same evaluation scheme applies to extra
prompts from R1 and extra sampling runs from R2.

In [ ]:
# TODO: for both prompts with expected != None, check whether
# p["expected"].lower() in <that prompt's greedy response>.lower()
# print id, expected, the check result, and the response itself
raise NotImplementedError("TODO: automatic evaluation")

In the markdown cell, evaluate every strategy run for the `paraphrase` prompt
(greedy, tau=0.7, and tau=1.3). Check whether the output preserves the original
meaning, is complete, and sounds natural. Each result needs a specific
justification rather than a general statement such as "looks good."

_(Answer space.)_

## 10. Diagnostic task

The following code **looks** correct. It raises no exception and generates
text, but it has a bug: `max_length` is confused with
`max_new_tokens`.

In [ ]:
DESIRED_NEW_TOKENS = 40
diag_prompt = "Write a short poem about the ocean."
diag_inputs = format_prompt(diag_prompt)  # noqa: F821
diag_prompt_len = diag_inputs["input_ids"].shape[1]
print(f"prompt_len={diag_prompt_len}, we want {DESIRED_NEW_TOKENS} NEW tokens")

with torch.no_grad():
    out_buggy = model.generate(
        **diag_inputs,
        max_length=DESIRED_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
actual_new_buggy = out_buggy.shape[1] - diag_prompt_len
print(f"BuggyGeneration: actual new tokens = {actual_new_buggy} (we wanted {DESIRED_NEW_TOKENS}!)")
text_buggy = tokenizer.decode(out_buggy[0, diag_prompt_len:], skip_special_tokens=True)
print(f"text (truncated): {text_buggy!r}")

Complete five steps:
1. Run the code with a short prompt and determine how many **new** tokens
   were actually produced, relative to how many the code's author
   intended to produce (the `DESIRED_NEW_TOKENS` variable).
2. Explain what exactly `max_length` counts and what
   does `max_new_tokens` count? (See the lecture, "The context window
   and truncation" section.)
3. Run the **same** code with a **longer** prompt (long enough that
   `prompt_len >= max_length`) and describe the result. The behavior should be
   different and more explicit than with the short prompt.
4. Fix the code by replacing `max_length=` with `max_new_tokens=`.
5. Verify the fix. For **both** prompt lengths, the number of newly
   generated tokens must match what was intended.

In [ ]:
# TODO:
# 1. as a comment: diagnose exactly what max_length counts here, and what we wanted
# 2. try the SAME buggy configuration with a LONGER prompt (prompt_len >= max_length)
#    - what happens now? (hint: a different, louder error)
# 3. fix: use max_new_tokens= instead of max_length=
# 4. verify the FIX for BOTH prompt lengths (assert on the number of new tokens)
raise NotImplementedError("TODO: diagnose and fix max_length/max_new_tokens")

## 11. Conclusions

**1. Which decoding strategy gave the most useful results for the unambiguous-answer tasks (extraction, format)? Where tested, did sampling help, hurt, or make no difference?**

_(Answer space.)_

**2. Did a higher temperature give subjectively better results for the qualitative task? Did diversity matter in this case?**

_(Answer space.)_

**3. What is the measured cost of the pipeline? How does it change with prompt length and with `use_cache`?**

_(Answer space.)_

**4. What specific limitations of the model or pipeline appeared during the experiment?**

_(Answer space.)_

**5. What follows from the trade-off among quality, cost, and configuration? Does a "better" configuration always produce a better result for every task type?**

_(Answer space.)_

---

## Extensions (optional, if time remains after sections 0-11)

- **R1.** Add 2 more prompts, such as a question with context and a text
  continuation, to the list from section 2. Apply the same
  treatment (sections 5, 6, 9) as the required prompts.
- **R2.** Run sampling with both temperatures for `format` and any R1
  prompts. The results show whether sampling helps, hurts, or makes no
  difference for unambiguous tasks excluded from the core comparison.
- Add a **sixth** original prompt in a different style from
  the suggestions above.
- Measure the effect of `repetition_penalty` on text continuation (does
  it reduce phrase repetition for longer generation?).

_(Optional extension. Code and results are not required for the core.)_